In [34]:
# import numpy as np, pandas as pd, optuna
# from catboost import CatBoostClassifier, Pool
# from sklearn.model_selection import StratifiedKFold, train_test_split
# from sklearn.metrics import roc_auc_score

# SEED = 42
# TUNE_SAMPLE = 300_000      # isi None untuk tuning di data penuh

# df = pd.read_csv("train_cleaned.csv")
# y = (df["Will_Buy_EV"] == "Yes").astype(int)
# X = df.drop(columns=["id", "Will_Buy_EV"])
# CAT_COLS = ["Gender", "City_Type", "Current_Car_Type",
#             "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]

In [35]:
# # 2a. Data untuk tuning: sampel stratified (atau data penuh)
# if TUNE_SAMPLE is None:
#     X_t, y_t = X, y
# else:
#     X_t, _, y_t, _ = train_test_split(
#         X, y, train_size=TUNE_SAMPLE, stratify=y, random_state=SEED
#     )

# # 2b. Satu holdout stratified 80/20 di dalam data tuning
# X_tr, X_va, y_tr, y_va = train_test_split(
#     X_t, y_t, test_size=0.2, stratify=y_t, random_state=SEED
# )
# print("Rasio Yes -> train:", round(y_tr.mean(), 4), "| valid:", round(y_va.mean(), 4))

# # Pool dibuat sekali di luar objective supaya tidak diulang tiap trial
# train_pool = Pool(X_tr, y_tr, cat_features=CAT_COLS)
# valid_pool = Pool(X_va, y_va, cat_features=CAT_COLS)

# # 2c. Optuna
# def objective(trial):
#     params = dict(
#         loss_function="Logloss",
#         eval_metric="AUC",
#         iterations=3000,
#         early_stopping_rounds=100,
#         learning_rate=trial.suggest_float("learning_rate", 0.05, 0.15, log=True),
#         depth=trial.suggest_int("depth", 4, 8),
#         l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 30, log=True),
#         random_strength=trial.suggest_float("random_strength", 0.1, 10, log=True),
#         bootstrap_type="Bernoulli",
#         subsample=trial.suggest_float("subsample", 0.6, 1.0),
#         rsm=trial.suggest_float("rsm", 0.6, 1.0),   # hapus baris ini jika pakai GPU
#         border_count=trial.suggest_categorical("border_count", [64, 128, 254]),
#         one_hot_max_size=trial.suggest_categorical("one_hot_max_size", [2, 4, 10]),
#         random_seed=SEED, verbose=0, thread_count=-1,
#     )
#     m = CatBoostClassifier(**params)
#     m.fit(train_pool, eval_set=valid_pool, use_best_model=True)
#     trial.set_user_attr("best_iter", int(m.get_best_iteration()))
#     return m.get_best_score()["validation"]["AUC"]

# study = optuna.create_study(direction="maximize",
#                             sampler=optuna.samplers.TPESampler(seed=SEED))
# study.optimize(objective, n_trials=20, timeout=90 * 60)   # berhenti di 20 trial atau 90 menit

# print("Best holdout AUC:", study.best_value)
# print("Best params     :", study.best_params)
# print("Best iteration  :", study.best_trial.user_attrs["best_iter"])

Best holdout AUC: 0.9419644141635528  
Best params     : {'learning_rate': 0.055710965281825744, 'depth': 4, 'l2_leaf_reg': 16.378726060483704, 'random_strength': 5.418648662113677, 'subsample': 0.930094231107212, 'rsm': 0.689140498436157, 'border_count': 254, 'one_hot_max_size': 4}  
Best iteration  : 1621  

In [36]:
# final_params = dict(
#     loss_function="Logloss", eval_metric="AUC",
#     iterations=5000, early_stopping_rounds=200,
#     bootstrap_type="Bernoulli",
#     random_seed=SEED, verbose=0, thread_count=-1,
#     **study.best_params,
# )

# skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# oof = np.zeros(len(X))
# fold_auc, best_iters = [], []

# for k, (tr, va) in enumerate(skf5.split(X, y), 1):
#     tr_pool = Pool(X.iloc[tr], y.iloc[tr], cat_features=CAT_COLS)
#     va_pool = Pool(X.iloc[va], y.iloc[va], cat_features=CAT_COLS)
#     m = CatBoostClassifier(**final_params)
#     m.fit(tr_pool, eval_set=va_pool, use_best_model=True)
#     oof[va] = m.predict_proba(va_pool)[:, 1]
#     fold_auc.append(roc_auc_score(y.iloc[va], oof[va]))
#     best_iters.append(m.get_best_iteration())
#     print(f"fold {k}: AUC={fold_auc[-1]:.5f}  best_iter={best_iters[-1]}")

# print(f"\nMean AUC = {np.mean(fold_auc):.5f} ± {np.std(fold_auc):.5f}")
# print(f"OOF AUC  = {roc_auc_score(y, oof):.5f}")

fold 1: AUC=0.94070  best_iter=2694  
fold 2: AUC=0.94157  best_iter=2719  
fold 3: AUC=0.94288  best_iter=2734  
fold 4: AUC=0.94254  best_iter=2843  
fold 5: AUC=0.94189  best_iter=2803  

Mean AUC = 0.94192 ± 0.00076  
OOF AUC  = 0.94191  

In [37]:
# import numpy as np, pandas as pd
# from catboost import CatBoostClassifier, Pool
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import roc_auc_score

# SEED = 42
# DATA_PATH = "train_cleaned.csv"
# NEW_TRAIN_PATH = "train_fe.csv"          # lokasi dataset train baru
# INCOME_FLOOR = 30000

# CAT_COLS = ["Gender", "City_Type", "Current_Car_Type",
#             "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]

# BEST_PARAMS = {'learning_rate': 0.055710965281825744, 'depth': 4, 'l2_leaf_reg': 16.378726060483704,
#                'random_strength': 5.418648662113677, 'subsample': 0.930094231107212,
#                'rsm': 0.689140498436157, 'border_count': 254, 'one_hot_max_size': 4}


# # ---------- 1. Fitur baru ----------
# def fit_fe_stats(train_df):
#     return {"city_income_mean": train_df.groupby("City_Type")["Annual_Income_USD"].mean().to_dict()}

# def add_features(df, stats):
#     out = df.copy()
#     inc = out["Annual_Income_USD"]
#     out["Income_Floor_Flag"] = (inc <= INCOME_FLOOR).astype(int)
#     out["Income_per_Car"]    = inc / out["Number_of_Cars_Owned"]
#     out["Income_vs_City"]    = inc / out["City_Type"].map(stats["city_income_mean"])
#     out["Multi_Car_Flag"]    = (out["Number_of_Cars_Owned"] >= 2).astype(int)
#     return out


# # ---------- 2. Data ----------
# df = pd.read_csv(DATA_PATH)
# target_raw = df.pop("Will_Buy_EV")               # simpan target asli (Yes/No)
# y = (target_raw == "Yes").astype(int)
# X = df.drop(columns="id")
# stats = fit_fe_stats(X)                          # simpan: dipakai lagi untuk test set
# X_fe = add_features(X, stats)                    # X + 4 fitur baru
# print("Kolom baru:", [c for c in X_fe.columns if c not in X.columns])


# # ---------- 3. Buat dataset train baru ----------
# train_fe = X_fe.copy()
# train_fe.insert(0, "id", df["id"].values)
# train_fe["Will_Buy_EV"] = target_raw.values      # target tetap Yes/No seperti aslinya

# assert train_fe.isna().sum().sum() == 0, "Ada NaN di dataset baru"
# assert np.isfinite(train_fe.select_dtypes("number")).all().all(), "Ada nilai inf di dataset baru"

# train_fe.to_csv(NEW_TRAIN_PATH, index=False)
# print("Dataset train baru:", train_fe.shape, "->", NEW_TRAIN_PATH)
# print(train_fe.head())


# # ---------- 4. Konfigurasi CV ----------
# final_params = dict(
#     loss_function="Logloss", eval_metric="AUC",
#     iterations=5000, early_stopping_rounds=200,
#     bootstrap_type="Bernoulli",
#     random_seed=SEED, verbose=0, thread_count=-1,
#     **BEST_PARAMS,
# )
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# folds = list(skf.split(X, y))                    # fold identik


# def cv_run(Xa, label):
#     oof = np.zeros(len(Xa)); aucs, iters = [], []
#     for k, (tr, va) in enumerate(folds, 1):
#         tr_pool = Pool(Xa.iloc[tr], y.iloc[tr], cat_features=CAT_COLS)
#         va_pool = Pool(Xa.iloc[va], y.iloc[va], cat_features=CAT_COLS)
#         m = CatBoostClassifier(**final_params)
#         m.fit(tr_pool, eval_set=va_pool, use_best_model=True)
#         oof[va] = m.predict_proba(va_pool)[:, 1]
#         aucs.append(roc_auc_score(y.iloc[va], oof[va]))
#         iters.append(m.get_best_iteration())
#         print(f"[{label}] fold {k}: AUC={aucs[-1]:.5f}  best_iter={iters[-1]}", flush=True)
#     print(f"[{label}] Mean AUC = {np.mean(aucs):.5f} ± {np.std(aucs):.5f} | "
#           f"OOF AUC = {roc_auc_score(y, oof):.5f}\n")
#     return aucs, oof


# # ---------- 5. Jalankan ----------
# fe_auc, fe_oof = cv_run(X_fe, "DENGAN FITUR")

Kolom baru: ['Income_Floor_Flag', 'Income_per_Car', 'Income_vs_City', 'Multi_Car_Flag']  
[DENGAN FITUR] fold 1: AUC=0.94088 best_iter=3087  
[DENGAN FITUR] fold 2: AUC=0.94179 best_iter=2647  
[DENGAN FITUR] fold 3: AUC=0.94309 best_iter=3339  
[DENGAN FITUR] fold 4: AUC=0.94271 best_iter=3226  
[DENGAN FITUR] fold 5: AUC=0.94207 best_iter=3455  
[DENGAN FITUR] Mean AUC = 0.94211 ± 0.00077 | OOF AUC = 0.94210  

In [38]:
import numpy as np, pandas as pd

TRAIN_FE_PATH = "train_fe.csv"      # dataset train yang sudah diberi 4 fitur baru
TEST_RAW_PATH = "test.csv"          # dataset test mentah
TEST_FE_PATH  = "test_fe.csv"       # hasil: test yang sudah sama seperti train
INCOME_FLOOR  = 30000

CAT_COLS = ["Gender", "City_Type", "Current_Car_Type",
            "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]

# ---------- referensi dari TRAIN ----------
train = pd.read_csv(TRAIN_FE_PATH)
train_cols = [c for c in train.columns if c != "Will_Buy_EV"]          # urutan kolom yang harus ditiru
city_income_mean = train.groupby("City_Type")["Annual_Income_USD"].mean().to_dict()

# ---------- 1. Baca & cek test mentah ----------
test = pd.read_csv(TEST_RAW_PATH)
print("Shape test mentah:", test.shape)
print("Missing per kolom :", test.isnull().sum()[test.isnull().sum() > 0].to_dict() or "tidak ada")
print("id duplikat       :", test["id"].duplicated().sum())

# ---------- 2. Cleaning (sama seperti train, TANPA menghapus baris) ----------
for c in CAT_COLS:
    test[c] = test[c].str.strip()                                       # buang spasi berlebih

for c in CAT_COLS:                                                       # kategori yang tidak ada di train
    unseen = set(test[c].dropna().unique()) - set(train[c].unique())
    if unseen:
        print(f"PERINGATAN {c}: kategori baru di test -> {unseen}")

range_checks = {"Age": (0, 120), "Annual_Income_USD": (0, None), "Daily_Commute_km": (0, None),
                "Number_of_Cars_Owned": (0, None), "Charging_Stations_Near_Home": (0, None),
                "Charging_Stations_Near_Work": (0, None), "Environmental_Concern_Level": (1, 5)}
for col, (lo, hi) in range_checks.items():
    bad = ((test[col] < lo) if lo is not None else False) | ((test[col] > hi) if hi is not None else False)
    if bad.sum() > 0:
        print(f"PERINGATAN {col}: {int(bad.sum())} nilai di luar rentang (baris tetap dipertahankan)")

# ---------- 3. Fitur baru (memakai statistik TRAIN) ----------
inc = test["Annual_Income_USD"]
test["Income_Floor_Flag"] = (inc <= INCOME_FLOOR).astype(int)
test["Income_per_Car"]    = inc / test["Number_of_Cars_Owned"]
test["Income_vs_City"]    = inc / test["City_Type"].map(city_income_mean)
test["Multi_Car_Flag"]    = (test["Number_of_Cars_Owned"] >= 2).astype(int)

# ---------- 4. Samakan dengan train & simpan ----------
test_fe = test[train_cols]                                               # urutan kolom identik dengan train
assert test_fe.isna().sum().sum() == 0, f"Masih ada NaN: {test_fe.isna().sum()[test_fe.isna().sum() > 0].to_dict()}"
assert np.isfinite(test_fe.select_dtypes("number")).all().all(), "Ada nilai inf"
assert len(test_fe) == len(pd.read_csv(TEST_RAW_PATH)), "Jumlah baris berubah"

test_fe.to_csv(TEST_FE_PATH, index=False)
print("Test baru:", test_fe.shape, "-> kolom sama dengan train:", list(test_fe.columns) == train_cols)

Shape test mentah: (286571, 14)
Missing per kolom : tidak ada
id duplikat       : 0
Test baru: (286571, 18) -> kolom sama dengan train: True


Shape test mentah: (286571, 14)  
Missing per kolom : tidak ada  
id duplikat       : 0  
Test baru: (286571, 18) -> kolom sama dengan train: True  

In [39]:
import os
import numpy as np, pandas as pd
from catboost import CatBoostClassifier, Pool

TRAIN_FE_PATH   = "train_fe.csv"
TEST_FE_PATH    = "test_fe.csv"
SAMPLE_SUB_PATH = "sample_submission.csv"   # format submit dari Kaggle (kalau ada di folder)
SUBMISSION_PATH = "submission.csv"
SEED = 42
ITERATIONS = 3500

CAT_COLS = ["Gender", "City_Type", "Current_Car_Type",
            "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]

BEST_PARAMS = {'learning_rate': 0.055710965281825744, 'depth': 4, 'l2_leaf_reg': 16.378726060483704,
               'random_strength': 5.418648662113677, 'subsample': 0.930094231107212,
               'rsm': 0.689140498436157, 'border_count': 254, 'one_hot_max_size': 4}

train = pd.read_csv(TRAIN_FE_PATH)
test  = pd.read_csv(TEST_FE_PATH)
y = (train.pop("Will_Buy_EV") == "Yes").astype(int)
X = train.drop(columns="id")
X_test = test.drop(columns="id")
assert list(X.columns) == list(X_test.columns), "Kolom train dan test tidak sama"

# latih di SEMUA data train (tanpa validation set -> jumlah iterasi tetap)
model = CatBoostClassifier(
    loss_function="Logloss", iterations=ITERATIONS,
    bootstrap_type="Bernoulli", random_seed=SEED,
    thread_count=-1, verbose=500, **BEST_PARAMS,
)
model.fit(Pool(X, y, cat_features=CAT_COLS))

# probabilitas kelas "Yes" (untuk ROC-AUC harus probabilitas, bukan label 0/1)
proba = model.predict_proba(Pool(X_test, cat_features=CAT_COLS))[:, 1]

# format submission
if os.path.exists(SAMPLE_SUB_PATH):
    sub = pd.read_csv(SAMPLE_SUB_PATH)
    id_col, target_col = sub.columns[0], sub.columns[1]
    sub[target_col] = pd.Series(proba, index=test["id"].values).reindex(sub[id_col].values).values
else:
    sub = pd.DataFrame({"id": test["id"], "Will_Buy_EV": proba})

assert sub.iloc[:, 1].notna().all(), "Ada prediksi kosong (id tidak cocok?)"
sub.to_csv(SUBMISSION_PATH, index=False)
print(sub.shape)
print(sub.head())

0:	learn: 0.6051309	total: 156ms	remaining: 9m 4s
500:	learn: 0.2276056	total: 1m 4s	remaining: 6m 24s
1000:	learn: 0.2256351	total: 2m 9s	remaining: 5m 23s
1500:	learn: 0.2245814	total: 3m 11s	remaining: 4m 14s
2000:	learn: 0.2236984	total: 4m 17s	remaining: 3m 13s
2500:	learn: 0.2229519	total: 5m 27s	remaining: 2m 10s
3000:	learn: 0.2222690	total: 6m 33s	remaining: 1m 5s
3499:	learn: 0.2216352	total: 7m 48s	remaining: 0us
(286571, 2)
       id  Will_Buy_EV
0  668665     0.009331
1  668666     0.020131
2  668667     0.006154
3  668668     0.002280
4  668669     0.021542
